# Python Decorators and Lambda Functions

## Part 1: Python Decorators

### 1. Introduction to Decorators

**What are Decorators?**
- Decorators are a powerful feature in Python that allows you to modify the behavior of functions or classes
- They are functions that take another function as an argument and return a new function
- Decorators allow you to wrap a function to extend its behavior without modifying the function itself
- They provide a clean way to add functionality (logging, timing, authentication, etc.) to existing code

**Key Concepts:**
1. Functions are first-class objects (can be passed as arguments)
2. Functions can be nested inside other functions
3. A function can return another function
4. Decorators use the `@` symbol syntax for cleaner code

### 2. Functions as First-Class Objects

In Python, functions are first-class objects, meaning they can be:
- Assigned to variables
- Passed as arguments to other functions
- Returned from other functions
- Stored in data structures

This is the foundation of how decorators work.

In [ ]:
# Functions as first-class objects
def greet(name):
    return f"Hello, {name}!"

# Assign function to a variable
my_function = greet
print("Function assigned to variable:")
print(my_function("Alice"))

# Pass function as argument
def call_function(func, arg):
    return func(arg)

print("\nFunction passed as argument:")
print(call_function(greet, "Bob"))

# Store function in a list
functions = [greet]
print("\nFunction stored in list:")
print(functions[0]("Charlie"))

### 3. Nested Functions and Closures

**Nested Functions**
Functions can be defined inside other functions. The inner function can access variables from the outer function.

**Closures**
A closure is a function that remembers the values from its enclosing scope even when the outer function has finished executing.

In [ ]:
# Nested functions
def outer_function(message):
    def inner_function():
        print(f"Message: {message}")
    return inner_function

print("Example 1: Closure in action")
my_func = outer_function("Hello from closure!")
my_func()  # Still remembers the message

print("\nExample 2: Multiple closures")
func1 = outer_function("First message")
func2 = outer_function("Second message")
func1()
func2()

### 4. Creating a Simple Decorator

A decorator is a function that:
1. Takes a function as an argument
2. Defines a wrapper function that adds functionality
3. Returns the wrapper function

**Syntax:**
```python
def decorator_name(func):
    def wrapper():
        # Code before function call
        result = func()
        # Code after function call
        return result
    return wrapper
```

In [ ]:
# Simple decorator example
def my_decorator(func):
    def wrapper():
        print("Before function call")
        result = func()
        print("After function call")
        return result
    return wrapper

@my_decorator
def say_hello():
    print("Hello!")

print("Using the decorated function:")
say_hello()

# This is equivalent to:
# say_hello = my_decorator(say_hello)

### 5. Decorators with Arguments

When the decorated function has arguments, the wrapper needs to accept and pass them through.

Use `*args` and `**kwargs` to handle any number of arguments.

In [ ]:
# Decorator with arguments
def decorator_with_args(func):
    def wrapper(*args, **kwargs):
        print(f"Calling function with args: {args}, kwargs: {kwargs}")
        result = func(*args, **kwargs)
        print(f"Function returned: {result}")
        return result
    return wrapper

@decorator_with_args
def add(a, b):
    return a + b

@decorator_with_args
def greet(name, greeting="Hello"):
    return f"{greeting}, {name}!"

print("Example 1: Function with positional args")
print(add(5, 3))

print("\nExample 2: Function with keyword args")
print(greet("Alice", greeting="Hi"))

print("\nExample 3: Function with mixed args")
print(greet("Bob"))

### 6. Practical Decorator Examples

**Timing Decorator**
Measure how long a function takes to execute.

**Logging Decorator**
Log function calls and their results.

**Memoization Decorator**
Cache function results to avoid redundant calculations.

In [ ]:
import time
import functools

# Timing decorator
def timer(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start_time = time.time()
        result = func(*args, **kwargs)
        end_time = time.time()
        print(f"{func.__name__} took {end_time - start_time:.4f} seconds")
        return result
    return wrapper

@timer
def slow_function():
    time.sleep(1)
    return "Done!"

print("Timing decorator example:")
print(slow_function())

In [ ]:
# Logging decorator
def logger(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        print(f"[LOG] Calling {func.__name__} with args={args}, kwargs={kwargs}")
        result = func(*args, **kwargs)
        print(f"[LOG] {func.__name__} returned {result}")
        return result
    return wrapper

@logger
def divide(a, b):
    if b == 0:
        raise ValueError("Cannot divide by zero")
    return a / b

print("Logging decorator example:")
print(divide(10, 2))
print()
print(divide(20, 4))

In [ ]:
# Memoization decorator (caching)
def memoize(func):
    cache = {}
    @functools.wraps(func)
    def wrapper(*args):
        if args not in cache:
            cache[args] = func(*args)
            print(f"Calculating {func.__name__}({args})")
        else:
            print(f"Using cached result for {args}")
        return cache[args]
    return wrapper

@memoize
def fibonacci(n):
    if n <= 1:
        return n
    return fibonacci(n - 1) + fibonacci(n - 2)

print("Memoization decorator example:")
print(f"fibonacci(10) = {fibonacci(10)}")
print()
print(f"fibonacci(10) = {fibonacci(10)}")  # Uses cache

### 7. Preserving Function Metadata

When you decorate a function, the wrapper replaces the original function. This can lose important metadata like the function's name, docstring, etc.

Use `@functools.wraps` to preserve the original function's metadata.

In [ ]:
import functools

# Without @functools.wraps
def decorator_without_wraps(func):
    def wrapper():
        return func()
    return wrapper

@decorator_without_wraps
def my_function():
    """This is my function."""
    return "Hello"

print("Without @functools.wraps:")
print(f"Function name: {my_function.__name__}")
print(f"Function docstring: {my_function.__doc__}")

# With @functools.wraps
def decorator_with_wraps(func):
    @functools.wraps(func)
    def wrapper():
        return func()
    return wrapper

@decorator_with_wraps
def my_function2():
    """This is my function."""
    return "Hello"

print("\nWith @functools.wraps:")
print(f"Function name: {my_function2.__name__}")
print(f"Function docstring: {my_function2.__doc__}")

### 8. Decorators with Custom Arguments

Sometimes you want decorators that accept their own arguments. This requires an extra layer of nesting.

**Syntax:**
```python
def decorator_with_arguments(arg1, arg2):
    def actual_decorator(func):
        def wrapper(*args, **kwargs):
            # Use arg1, arg2 here
            return func(*args, **kwargs)
        return wrapper
    return actual_decorator
```

In [ ]:
# Decorator with custom arguments
def repeat(times):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            results = []
            for _ in range(times):
                results.append(func(*args, **kwargs))
            return results
        return wrapper
    return decorator

@repeat(3)
def greet(name):
    return f"Hello, {name}!"

print("Decorator with arguments example:")
print(greet("Alice"))

@repeat(5)
def say_hi():
    return "Hi!"

print(say_hi())

### 9. Class Decorators

Decorators can also be implemented as classes. The class must implement `__init__` and `__call__` methods.

In [ ]:
# Class decorator example
class CountCalls:
    def __init__(self, func):
        self.func = func
        self.count = 0
        functools.update_wrapper(self, func)
    
    def __call__(self, *args, **kwargs):
        self.count += 1
        print(f"Call {self.count} of {self.func.__name__}")
        return self.func(*args, **kwargs)

@CountCalls
def say_hello():
    return "Hello!"

print("Class decorator example:")
print(say_hello())
print(say_hello())
print(say_hello())
print(f"Total calls: {say_hello.count}")

### 10. Built-in Decorators

Python provides several built-in decorators:

**@staticmethod**
Defines a method that doesn't need access to instance or class.

**@classmethod**
Defines a method that receives the class as first argument.

**@property**
Defines a getter for a class attribute.

In [ ]:
# Built-in decorators example
class MyClass:
    class_variable = "I am a class variable"
    
    def __init__(self, value):
        self.value = value
    
    @staticmethod
    def static_method():
        return "Static method called"
    
    @classmethod
    def class_method(cls):
        return f"Class method called. Class variable: {cls.class_variable}"
    
    @property
    def value_property(self):
        return f"Value: {self.value}"

print("Built-in decorators example:")
print(MyClass.static_method())
print(MyClass.class_method())

obj = MyClass(42)
print(obj.value_property)

### 11. Chaining Decorators

You can apply multiple decorators to a single function. They are applied from bottom to top.

In [ ]:
# Chaining decorators
def make_bold(func):
    @functools.wraps(func)
    def wrapper():
        return f"<b>{func()}</b>"
    return wrapper

def make_italic(func):
    @functools.wraps(func)
    def wrapper():
        return f"<i>{func()}</i>"
    return wrapper

@make_bold
@make_italic
def greet():
    return "Hello"

print("Chaining decorators example:")
print(greet())
# Equivalent to: greet = make_bold(make_italic(greet))

---

## Part 2: Python Lambda Functions

### 12. Introduction to Lambda Functions

**What are Lambda Functions?**
- Lambda functions are small, anonymous functions defined with the `lambda` keyword
- They can have any number of arguments but only one expression
- The expression is evaluated and returned when the lambda is called
- Also known as "anonymous functions" because they don't have a name
- Useful for short, simple operations where a full function definition would be overkill

**Key Characteristics:**
1. Anonymous (no function name)
2. Single expression (no statements)
3. Can take multiple arguments
4. Returns the result of the expression
5. Defined in one line

**Syntax:**
```python
lambda arguments: expression
```

### 13. Basic Lambda Function Syntax

Compare a regular function with a lambda function that does the same thing.

In [ ]:
# Regular function vs Lambda function

# Regular function
def add_regular(a, b):
    return a + b

# Lambda function
add_lambda = lambda a, b: a + b

print("Regular function:")
print(f"add_regular(5, 3) = {add_regular(5, 3)}")

print("\nLambda function:")
print(f"add_lambda(5, 3) = {add_lambda(5, 3)}")

print("\nType check:")
print(f"Type of add_regular: {type(add_regular)}")
print(f"Type of add_lambda: {type(add_lambda)}")

### 14. Lambda with Single and Multiple Arguments

Lambda functions can accept single or multiple arguments.

In [ ]:
# Lambda with single argument
square = lambda x: x ** 2
cube = lambda x: x ** 3
increment = lambda x: x + 1

print("Single argument lambda examples:")
print(f"square(5) = {square(5)}")
print(f"cube(3) = {cube(3)}")
print(f"increment(10) = {increment(10)}")

# Lambda with multiple arguments
add = lambda a, b: a + b
multiply = lambda a, b: a * b
power = lambda base, exponent: base ** exponent

print("\nMultiple argument lambda examples:")
print(f"add(10, 20) = {add(10, 20)}")
print(f"multiply(5, 6) = {multiply(5, 6)}")
print(f"power(2, 8) = {power(2, 8)}")

### 15. Lambda with Default Arguments

Lambda functions can have default argument values just like regular functions.

In [ ]:
# Lambda with default arguments
greet = lambda name, greeting="Hello": f"{greeting}, {name}!"
power = lambda base, exponent=2: base ** exponent

print("Lambda with default arguments:")
print(f"greet('Alice') = {greet('Alice')}")
print(f"greet('Bob', 'Hi') = {greet('Bob', 'Hi')}")
print(f"power(5) = {power(5)}")
print(f"power(5, 3) = {power(5, 3)}")

### 16. Lambda with Built-in Functions

Lambda functions are commonly used with built-in functions like `map()`, `filter()`, and `reduce()`.

In [ ]:
# Lambda with map()
numbers = [1, 2, 3, 4, 5]

# Square each number
squared = list(map(lambda x: x ** 2, numbers))
print(f"Original: {numbers}")
print(f"Squared: {squared}")

# Add 10 to each number
added = list(map(lambda x: x + 10, numbers))
print(f"Added 10: {added}")

In [ ]:
# Lambda with filter()
numbers = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

# Filter even numbers
evens = list(filter(lambda x: x % 2 == 0, numbers))
print(f"Original: {numbers}")
print(f"Even numbers: {evens}")

# Filter numbers greater than 5
greater_than_five = list(filter(lambda x: x > 5, numbers))
print(f"Greater than 5: {greater_than_five}")

In [ ]:
from functools import reduce

# Lambda with reduce()
numbers = [1, 2, 3, 4, 5]

# Sum of all numbers
total = reduce(lambda x, y: x + y, numbers)
print(f"Original: {numbers}")
print(f"Sum: {total}")

# Product of all numbers
product = reduce(lambda x, y: x * y, numbers)
print(f"Product: {product}")

# Find maximum
maximum = reduce(lambda x, y: x if x > y else y, numbers)
print(f"Maximum: {maximum}")

### 17. Lambda with sorted()

Lambda functions are useful for custom sorting with the `key` parameter.

In [ ]:
# Lambda with sorted()
students = [
    {'name': 'Alice', 'age': 25},
    {'name': 'Bob', 'age': 20},
    {'name': 'Charlie', 'age': 30},
    {'name': 'David', 'age': 22}
]

# Sort by age
sorted_by_age = sorted(students, key=lambda x: x['age'])
print("Sorted by age:")
for student in sorted_by_age:
    print(f"  {student['name']}: {student['age']}")

# Sort by name
sorted_by_name = sorted(students, key=lambda x: x['name'])
print("\nSorted by name:")
for student in sorted_by_name:
    print(f"  {student['name']}: {student['age']}")

In [ ]:
# Sorting tuples by specific element
points = [(1, 2), (4, 1), (3, 5), (2, 3)]

# Sort by second element
sorted_by_second = sorted(points, key=lambda x: x[1])
print(f"Original: {points}")
print(f"Sorted by second element: {sorted_by_second}")

# Sort by sum of elements
sorted_by_sum = sorted(points, key=lambda x: x[0] + x[1])
print(f"Sorted by sum: {sorted_by_sum}")

### 18. Lambda with Conditional Expressions

You can use conditional expressions (ternary operator) in lambda functions.

In [ ]:
# Lambda with conditional expressions
get_grade = lambda score: 'A' if score >= 90 else 'B' if score >= 80 else 'C' if score >= 70 else 'D' if score >= 60 else 'F'

print("Grade examples:")
print(f"Score 95: {get_grade(95)}")
print(f"Score 85: {get_grade(85)}")
print(f"Score 75: {get_grade(75)}")
print(f"Score 65: {get_grade(65)}")
print(f"Score 55: {get_grade(55)}")

# Absolute value
absolute = lambda x: x if x >= 0 else -x
print(f"\nAbsolute value of -5: {absolute(-5)}")
print(f"Absolute value of 5: {absolute(5)}")

### 19. Lambda Immediately Invoked

Lambda functions can be defined and immediately called (IIFE - Immediately Invoked Function Expression).

In [ ]:
# Immediately invoked lambda
result = (lambda x, y: x + y)(5, 10)
print(f"Immediately invoked lambda: {result}")

# Another example
greeting = (lambda name: f"Hello, {name}!")("Alice")
print(f"Greeting: {greeting}")

# Useful for one-time calculations
discounted_price = (lambda price, discount: price * (1 - discount))(100, 0.2)
print(f"Discounted price: ${discounted_price:.2f}")

### 20. Lambda with Dictionary Operations

Lambda functions are useful for dictionary operations like sorting and filtering.

In [ ]:
# Lambda with dictionary
students = {
    'Alice': 85,
    'Bob': 92,
    'Charlie': 78,
    'David': 88
}

# Sort dictionary by values
sorted_by_value = dict(sorted(students.items(), key=lambda x: x[1]))
print(f"Original: {students}")
print(f"Sorted by value: {sorted_by_value}")

# Sort dictionary by keys
sorted_by_key = dict(sorted(students.items(), key=lambda x: x[0]))
print(f"Sorted by key: {sorted_by_key}")

# Filter dictionary
high_scorers = dict(filter(lambda x: x[1] >= 85, students.items()))
print(f"High scorers (>=85): {high_scorers}")

### 21. Lambda Limitations

**What Lambda CANNOT do:**
- Cannot contain multiple statements
- Cannot contain assignments
- Cannot use `return`, `yield`, `pass`, `assert`
- Cannot have docstrings
- Cannot have type hints (in Python < 3.12)

**When to Use Lambda:**
✅ Short, simple operations
✅ One-time use functions
✅ Callback functions
✅ With map, filter, reduce, sorted

**When NOT to Use Lambda:**
❌ Complex logic
❌ Multiple statements needed
❌ Reusable functions
❌ When readability is important

In [ ]:
# Example of what lambda CANNOT do

# This is INVALID - lambda cannot contain assignments
# invalid_lambda = lambda x: y = x * 2; y + 1

# This is INVALID - lambda cannot contain multiple statements
# invalid_lambda = lambda x: print(x); x * 2

# This is VALID - single expression
valid_lambda = lambda x: x * 2 + 1
print(f"Valid lambda: {valid_lambda(5)}")

# For complex logic, use a regular function
def complex_operation(x):
    temp = x * 2
    result = temp + 1
    return result

print(f"Regular function: {complex_operation(5)}")

### 22. Practical Examples

Real-world use cases for lambda functions.

In [ ]:
# Practical Example 1: Data transformation
products = [
    {'name': 'Laptop', 'price': 1000},
    {'name': 'Mouse', 'price': 25},
    {'name': 'Keyboard', 'price': 75}
]

# Apply 10% discount
discounted = list(map(lambda p: {**p, 'price': p['price'] * 0.9}, products))
print("Original prices:")
for p in products:
    print(f"  {p['name']}: ${p['price']}")
print("\nAfter 10% discount:")
for p in discounted:
    print(f"  {p['name']}: ${p['price']:.2f}")

In [ ]:
# Practical Example 2: String operations
words = ['python', 'lambda', 'function', 'programming']

# Capitalize first letter
capitalized = list(map(lambda w: w.capitalize(), words))
print(f"Original: {words}")
print(f"Capitalized: {capitalized}")

# Filter long words
long_words = list(filter(lambda w: len(w) > 5, words))
print(f"Long words (>5 chars): {long_words}")

# Sort by length
sorted_by_length = sorted(words, key=lambda w: len(w))
print(f"Sorted by length: {sorted_by_length}")

---

## Summary

### Decorators Summary

| Concept | Description | Example |
|---------|-------------|--------|
| First-class functions | Functions can be passed as arguments | `def decorator(func):` |
| Nested functions | Functions inside functions | `def outer(): def inner():` |
| Closures | Inner functions remember outer scope | `func = outer(); func()` |
| Basic decorator | Wraps a function | `@decorator` |
| Decorator with args | Handles function arguments | `def wrapper(*args, **kwargs):` |
| functools.wraps | Preserves function metadata | `@functools.wraps(func)` |
| Class decorator | Uses __call__ method | `class Decorator:` |
| Chaining | Multiple decorators on one function | `@dec1 @dec2 def func():` |

### Lambda Functions Summary

| Concept | Syntax | Example |
|---------|--------|--------|
| Basic lambda | `lambda x: expression` | `lambda x: x * 2` |
| Multiple args | `lambda x, y: expression` | `lambda a, b: a + b` |
| Default args | `lambda x, y=1: expression` | `lambda x, y=1: x * y` |
| With map | `map(lambda x: x*2, list)` | `list(map(lambda x: x**2, nums))` |
| With filter | `filter(lambda x: condition, list)` | `list(filter(lambda x: x>5, nums))` |
| With sorted | `sorted(list, key=lambda x: x.attr)` | `sorted(students, key=lambda x: x.age)` |
| Conditional | `lambda x: a if cond else b` | `lambda x: 'even' if x%2==0 else 'odd'` |

### Key Takeaways

**Decorators:**
- Modify function behavior without changing the function itself
- Use `@functools.wraps` to preserve metadata
- Common use cases: logging, timing, authentication, caching
- Can be chained together

**Lambda Functions:**
- Anonymous, single-expression functions
- Ideal for short, simple operations
- Commonly used with map, filter, sorted, reduce
- Cannot contain statements or multiple expressions
- Use regular functions for complex logic and reusability